In [ ]:
# Cell 1 — configure HPC sources and a NEW diagnostic output directory.
from pathlib import Path
import hashlib,json,platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
V12=Path('/data1/huangy/fig6/NC/v12/four_init_full_pipeline_NCVIconcurrent')
NCVI_RUN=V12/'ncvi_daily_standalone/run_001'
RAW_DAILY=V12/'tables/table_daily_member_raw_factors_SHF.csv'
TCC_DAILY=V12/'tables/table_daily_member_tcc_SHF.csv'
TMAX_DAILY=V12/'tables/table_daily_tmax_SHF.csv'
STAGE=V12/'tables/table_stage_member_raw_factors_TCC.csv'
MODEL=V12/'mlr_raw_residual_comparison'
# Optional existing ERA5 physical reference only; NCVI is ALWAYS excluded.
# Set None to explicitly omit all ERA5 reference lines.
ERA5_PHYSICAL=Path('/data1/huangy/fig6/NC/v11/hotcold_stage_group_timeseries_tables_multiinit_totalfig6/tables/daily_timeseries_all_factors.csv')
OUTPUT=V12/'process_preliminary/run_001'
if OUTPUT.exists():raise ValueError('Choose a NEW output directory')
if OUTPUT.resolve()==NCVI_RUN.resolve() or NCVI_RUN.resolve() in OUTPUT.resolve().parents:raise ValueError('Output must be outside immutable NCVI run')
INITS=('2023-06-08','2023-06-12');GROUPS=('Hot17','Middle17','Cold17')
VERSIONS=('residualized','raw')
FACTORS5=('NCHN_tp','z500_anom_NCHN','sm_avg','NCVI_concurrent','TCC_Avg')
FACTORS11=('sm_avg','z500_anom_NCHN','SHF_Avg','NCVI_concurrent','TCC_Avg')
PREDICTORS={'residualized':('sm_avg','SHF_resid','z500_anom_NCHN','Initial_MSEstar_max_NCHN','TCC_resid','NCVI_concurrent'),
'raw':('sm_avg','SHF_Avg','z500_anom_NCHN','Initial_MSEstar_max_NCHN','TCC_Avg','NCVI_concurrent')}
FACTORS=('y_tmax','NCHN_tp','sm_avg','SHF_Avg','z500_anom_NCHN','TCC_Avg');KEY=['init_date','date','member']
SNAPSHOTS={}
def capture(number):
 text=get_ipython().history_manager.input_hist_raw[-1]
 if not text.strip():raise RuntimeError('Missing execution snapshot')
 SNAPSHOTS[number]=text
 for k in list(SNAPSHOTS):
  if k>number:del SNAPSHOTS[k]
capture(1)
print('Cell 1 READY; PRELIMINARY / DIAGNOSTIC only; ERA5 NCVI hidden')


In [ ]:
# Cell 2 — independent physical-source and saved-model verification.
capture(2)
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def require(df, names, context):
    missing = set(names) - set(df.columns)
    if missing:
        raise ValueError(f'{context}: missing fields {sorted(missing)}')

def finite(values, context):
    a = np.asarray(values, float)
    if not np.isfinite(a).all():
        raise ValueError(f'{context}: missing or nonfinite values; no rows dropped')
    return a

def normalize_window(series):
    return series.replace({'Stage-I_dry': 'Stage-I', 'Stage_I': 'Stage-I'})

def parameters(data, init, version):
    """Load a complete saved Stage-I model and validate its 51-member design."""
    summary, coef, design = data['summary'], data['coefficients'], data['design']
    require(summary, ['init_date', 'window', 'version', 'n', 'design_rank', 'predictor_list', 'standardized_intercept'], 'model summary')
    require(coef, ['init_date', 'window', 'version', 'predictor', 'position', 'standardized_coef', 'mean', 'scale_ddof0'], 'coefficients')
    require(design, ['init_date', 'window', 'version', 'member', 'predictor', 'value', 'y_tmax'], 'training design')
    select = lambda d: d[(d.init_date == init) & (d.window == 'Stage-I') & (d.version == version)].copy()
    s = select(summary)
    c = select(coef)
    x = select(design)
    if len(s) != 1 or s.iloc[0]['n'] != 51 or s.iloc[0].design_rank != 7 or s.iloc[0].predictor_list != ';'.join(PREDICTORS[version]):
        raise ValueError(f'{init}/{version}: invalid complete final Stage-I model')
    c = c[c.predictor != 'const'].sort_values('position')
    if c.predictor.tolist() != list(PREDICTORS[version]) or c.position.tolist() != list(range(1, 7)):
        raise ValueError(f'{init}/{version}: coefficients are not the final six predictors')
    finite(c[['standardized_coef', 'mean', 'scale_ddof0']], 'saved scaler/coefficients')
    if (c.scale_ddof0 <= 0).any():
        raise ValueError('Saved scaler must have positive scales')
    if len(x) != 306 or x.duplicated(['member', 'predictor']).any():
        raise ValueError(f'{init}/{version}: incomplete training design')
    for row in c.itertuples():
        q = x[x.predictor == row.predictor].sort_values('member')
        if not np.array_equal(q.member.to_numpy(), np.arange(51)):
            raise ValueError('Training design member coverage mismatch')
        a = finite(q.value, 'training predictors')
        if not np.isclose(a.mean(), row.mean, rtol=0, atol=1e-8) or not np.isclose(a.std(ddof=0), row.scale_ddof0, rtol=0, atol=1e-8):
            raise ValueError('Training design does not reproduce saved scaler')
    pivot = x.pivot(index='member', columns='predictor', values='value').loc[:, list(PREDICTORS[version])]
    y = x.pivot(index='member', columns='predictor', values='y_tmax')
    finite(y, 'training y_tmax')
    if not (y.to_numpy() == y.iloc[:, 0].to_numpy()[:, None]).all():
        raise ValueError('Training response differs by predictor position')
    intercept = float(s.iloc[0].standardized_intercept)
    finite(intercept, 'standardized intercept')
    fitted = select(data['fitted']).sort_values('member')
    require(fitted, ['member','y_tmax','y_fitted','residual'], 'saved fitted members')
    if len(fitted) != 51 or not np.array_equal(fitted.member, np.arange(51)):
        raise ValueError('Saved fitted members incomplete')
    reconstructed = intercept + ((pivot.to_numpy() - c['mean'].to_numpy()) /
                                  c.scale_ddof0.to_numpy()) @ c.standardized_coef.to_numpy()
    finite(fitted[['y_tmax','y_fitted','residual']], 'saved fitted members')
    if not np.allclose(reconstructed, fitted.y_fitted, atol=1e-6, rtol=0) or not np.allclose(y.iloc[:,0], fitted.y_tmax, atol=1e-8, rtol=0):
        raise ValueError('Full-model replay differs from saved fit/Stage-I training response; coefficients are not replaced')
    if not np.allclose(fitted.y_tmax-fitted.y_fitted, fitted.residual, atol=1e-8, rtol=0):
        raise ValueError('Saved fitted residual identity failed')
    mean_state_prediction = float(intercept + ((pivot.mean().to_numpy()-c['mean'].to_numpy()) / c.scale_ddof0.to_numpy()) @ c.standardized_coef.to_numpy())
    if not np.isclose(mean_state_prediction, intercept, atol=1e-8, rtol=0) or not np.isclose(mean_state_prediction, fitted.y_fitted.mean(), atol=1e-8, rtol=0):
        raise ValueError('Diagnostic baseline does not equal full-model prediction at training mean state')
    return {'intercept': intercept, 'coef': c.set_index('predictor'),
            'design': pivot, 'y': y.iloc[:, 0].sort_index(), 'version': version,
            'mean_state_prediction': mean_state_prediction,
            'full_fit_max_abs_difference': float(np.max(np.abs(reconstructed-fitted.y_fitted.to_numpy())))}

def residual_relation(data, init):
    a = data['residualization']
    require(a, ['init_date', 'window', 'target', 'source', 'control', 'intercept', 'slope'], 'residualization audit')
    a = a[(a.init_date == init) & (a.window == 'Stage-I') & (a.target == 'SHF_resid')]
    if len(a) != 1 or a.iloc[0].source != 'SHF_Avg' or a.iloc[0].control != 'sm_avg':
        raise ValueError(f'{init}: require saved Stage-I SHF_Avg ~ intercept + sm_avg residual relation')
    return tuple(finite(a[['intercept', 'slope']], 'fixed residual relation')[0])

def partial_fitted_tmax(sm_value, shf_value, model, relation):
    """Actual diagnostic entry: alpha_std + beta_SM*z_SM + beta_SHF*z_SHF.

    All other six-factor full-model predictors remain at their own training means.
    Saved stage-member residual parameters are used without any daily fitting.
    """
    smv, shfv = np.asarray(sm_value, float), np.asarray(shf_value, float)
    factor = 'SHF_resid' if model['version'] == 'residualized' else 'SHF_Avg'
    projected = shfv - (relation[0] + relation[1] * smv) if factor == 'SHF_resid' else shfv
    c = model['coef']
    value = (model['intercept'] + c.loc['sm_avg', 'standardized_coef'] *
             (smv - c.loc['sm_avg', 'mean']) / c.loc['sm_avg', 'scale_ddof0'] +
             c.loc[factor, 'standardized_coef'] *
             (projected - c.loc[factor, 'mean']) / c.loc[factor, 'scale_ddof0'])
    return value, projected
def adapt_v12_daily(raw, tcc, tmax, stage):
    """Return physical long data, current groups and daily-to-stage numeric audit.

    Dates retain source labels: TCC UTC interval metadata is preserved and
    explicitly audited separately, rather than silently shifted to BJT days.
    """
    frames = []
    dates = pd.date_range('2023-06-14', '2023-06-24')
    expected = pd.MultiIndex.from_product([INITS, dates, range(51)], names=KEY)
    for name, source in [('raw', raw), ('tcc', tcc), ('tmax', tmax)]:
        d = source[source.init_date.isin(INITS)].copy()
        d['date'] = pd.to_datetime(d.date, errors='raise')
        if d.duplicated(KEY).any() or not pd.MultiIndex.from_frame(d[KEY]).sort_values().equals(expected.sort_values()):
            raise ValueError(f'{name}: require exactly 2 inits x 11 dates x 51 integer members')
        frames.append(d)
    d, c, t = frames
    d = d.merge(c[KEY + ['TCC_Avg']], on=KEY, validate='one_to_one')
    check = d.merge(t[KEY + ['s2s_tmax']], on=KEY, validate='one_to_one')
    if not np.array_equal(check.y_tmax.to_numpy(), check.s2s_tmax.to_numpy()):
        raise ValueError('Daily Tmax tables differ')
    finite(d[list(FACTORS)], 'v12 physical factors')
    s = stage.copy(); s['window'] = normalize_window(s.stage)
    groups, audit, pieces = [], [], []
    for init in INITS:
        for window, end in [('Stage-I', '2023-06-17'), ('Total', '2023-06-24')]:
            target = s[(s.init_date == init) & (s.window == window)].set_index('member').sort_index()
            if len(target) != 51 or not np.array_equal(target.index, np.arange(51)):
                raise ValueError('Stage input member coverage invalid')
            ranking = target.y_tmax.sort_values(ascending=False)
            if ranking.iloc[16] == ranking.iloc[17] or ranking.iloc[33] == ranking.iloc[34]:
                raise ValueError('Boundary tie requires explicit scientific ranking rule')
            g = pd.DataFrame({'member': ranking.index, 'rank': range(1,52)})
            g['group'] = [GROUPS[i//17] for i in range(51)]
            g['init_date'], g['window'] = init, window
            groups.append(g)
            q = d[(d.init_date == init) & (d.date <= end)]
            for factor in FACTORS:
                rebuilt = q.groupby('member')[factor].sum() if factor == 'NCHN_tp' else q.groupby('member')[factor].mean()
                for member in range(51):
                    delta = float(rebuilt.loc[member] - target.loc[member, factor])
                    audit.append(dict(init_date=init, window=window, member=member, factor=factor,
                                      daily_aggregate=rebuilt.loc[member], stage_value=target.loc[member,factor],
                                      difference=delta, aggregation='sum' if factor=='NCHN_tp' else 'mean',
                                      passed=abs(delta)<=1e-8))
            display = d[d.init_date == init].copy()
            display = display.merge(g[['member','group']], on='member', validate='many_to_one')
            display['window'] = window
            long = display.melt(id_vars=KEY+['window','group'], value_vars=list(FACTORS), var_name='factor', value_name='value')
            pieces.append(long)
    qc = pd.DataFrame(audit)
    if not qc.passed.all():
        raise ValueError('v12 daily-to-stage consistency FAILED; do not export')
    return pd.concat(pieces, ignore_index=True), pd.concat(groups, ignore_index=True), qc

def group_daily(long):
    """Group means use current v12 membership, never legacy grouped curves."""
    keys = ['init_date','window','date','group','factor']
    counts = long.groupby(keys).size()
    if not (counts == 17).all():
        raise ValueError('Each physical group requires exactly 17 members')
    return long.groupby(keys, as_index=False).value.mean()

inputs={'raw_daily':RAW_DAILY,'tcc_daily':TCC_DAILY,'tmax_daily':TMAX_DAILY,'stage':STAGE,
'comparison_manifest':MODEL/'qc/comparison_manifest.json'}
for key,name in {'summary':'table_mlr_model_summary.csv','coefficients':'table_mlr_model_coefficients.csv',
'design':'table_mlr_design_members.csv','fitted':'table_mlr_fitted_members.csv','residualization':'table_mlr_residualization_audit.csv'}.items():inputs[key]=MODEL/'tables'/name
hashes={k:{'path':str(p),'sha256':sha256(p)} for k,p in inputs.items()}
mm=json.loads(inputs['comparison_manifest'].read_text())
for key in ['summary','coefficients','design','fitted','residualization']:
 if hashes[key]['sha256']!=mm['table_sha256'][inputs[key].name]:raise ValueError('Model hash mismatch')
if hashes['stage']['sha256']!=next(x['sha256'] for x in mm['provenance'] if x['name']=='raw'):raise ValueError('Stage source hash mismatch')
data={k:pd.read_csv(inputs[k]) for k in ['summary','coefficients','design','fitted','residualization']}
physical_long,current_groups,physical_qc=adapt_v12_daily(pd.read_csv(RAW_DAILY),pd.read_csv(TCC_DAILY),pd.read_csv(TMAX_DAILY),pd.read_csv(STAGE))
physical_group=group_daily(physical_long)
print('Cell 2 PASS; v12 grouping and 1224 daily-to-stage checks; max error',physical_qc.difference.abs().max())


In [ ]:
# Cell 3 — authorize diagnostic use only; preserve original NCVI FAIL.
capture(3)
original_run_hashes={str(p.relative_to(NCVI_RUN)):sha256(p) for p in NCVI_RUN.rglob('*') if p.is_file()}
nm=json.loads((NCVI_RUN/'ncvi_daily_export_manifest.json').read_text())
for name,digest in nm['outputs'].items():
 if Path(name).name!=name or original_run_hashes.get(name)!=digest:raise ValueError('NCVI run hash mismatch')
if nm['s2s_daily_source_qc']['status']!='PASS':raise ValueError('S2S physical source must PASS')
ncvi=pd.read_csv(NCVI_RUN/'table_ncvi_concurrent_daily_members.csv');ncvi['date']=pd.to_datetime(ncvi.date)
expected=pd.MultiIndex.from_product([INITS,pd.date_range('2023-06-14','2023-06-24'),range(51)],names=KEY)
if len(ncvi)!=1122 or ncvi.duplicated(KEY).any() or not pd.MultiIndex.from_frame(ncvi[KEY]).sort_values().equals(expected.sort_values()) or not ncvi.units.eq('PVU').all() or not ncvi.anomaly_used.eq(True).all():raise ValueError('NCVI coverage/units/anomaly invalid')
finite(ncvi.daily_ncvi_anomaly_pvu,'actual daily anomaly')
rq=pd.read_csv(NCVI_RUN/'table_ncvi_daily_stage_replay_qc.csv')
if len(rq)!=204 or rq.duplicated(['init_date','window','member']).any():raise ValueError('NCVI stage QC coverage invalid')
internal_error=max((rq.daily_equal_day_aggregate-rq.mlr_scalar_replay).abs().max(),(rq.daily_time_weighted_aggregate-rq.mlr_scalar_replay).abs().max())
if not np.isfinite(internal_error) or internal_error>1e-8:raise ValueError('Daily INTERNAL aggregation fails')
pieces=[]
for window in ['Stage-I','Total']:
 g=current_groups[current_groups.window==window]
 q=ncvi.merge(g[['init_date','member','group']],on=['init_date','member'],validate='many_to_one')
 if len(q)!=1122 or q.group.isna().any():raise ValueError('Group mapping failed')
 q['window']=window;q['factor']='NCVI_concurrent';q['value']=q.daily_ncvi_anomaly_pvu
 pieces.append(q[KEY+['window','group','factor','value']])
ng=group_daily(pd.concat(pieces,ignore_index=True))
all_physical=pd.concat([physical_group,ng],ignore_index=True)
# ERA5 physical references are independent existing observations, never S2S group means.
if ERA5_PHYSICAL is not None:
 ed=pd.read_csv(ERA5_PHYSICAL);hashes['era5_physical']={'path':str(ERA5_PHYSICAL),'sha256':sha256(ERA5_PHYSICAL)}
 erows=[]
 for init in INITS:
  for window,end,factors in [('Total','2023-06-24',FACTORS5),('Stage-I','2023-06-18',FACTORS11)]:
   for factor in factors:
    if factor=='NCVI_concurrent':continue
    z=ed[(ed.init==init)&(ed.grouping_stage==window)&(ed.factor==factor)].copy();z['date']=pd.to_datetime(z.date);z=z[z.date.between('2023-06-14',end)].sort_values('date')
    if len(z)!=len(pd.date_range('2023-06-14',end)) or z.date.duplicated().any():raise ValueError('ERA5 physical date coverage invalid')
    finite(z.era5,'ERA5 physical values')
    for r in z.itertuples():erows.append(dict(init_date=init,window=window,date=r.date,group='ERA5',factor=factor,value=r.era5))
 all_physical=pd.concat([all_physical,pd.DataFrame(erows)],ignore_index=True)
fig5_table=all_physical[(all_physical.window=='Total')&all_physical.factor.isin(FACTORS5)].copy()
fig11_physical=all_physical[(all_physical.window=='Stage-I')&all_physical.factor.isin(FACTORS11)&(all_physical.date<=pd.Timestamp('2023-06-18'))].copy()
print('Cell 3 DIAGNOSTIC use ready; internal max',internal_error,'historical status preserved:',nm['status'],'ERA5 NCVI OMITTED')


In [ ]:
# Cell 4 — frozen saved-model partial fits; physical series identical by version.
capture(4)
partial=[];audit=[]
for init in INITS:
 for version in VERSIONS:
  model=parameters(data,init,version);relation=residual_relation(data,init)
  s=fig11_physical[(fig11_physical.init_date==init)&(fig11_physical.group!='ERA5')].pivot(index=['date','group'],columns='factor',values='value').reset_index()
  vals,heat=partial_fitted_tmax(s.sm_avg,s.SHF_Avg,model,relation)
  for j,r in enumerate(s.itertuples()):partial.append(dict(init_date=init,window='Stage-I',date=r.date,group=r.group,factor='partial_fitted_Tmax',value=vals[j],model_version=version,outside_training_window=r.date>pd.Timestamp('2023-06-17'),projected_heat=heat[j]))
  audit.append(dict(init_date=init,model_version=version,intercept=model['intercept'],fixed_residual_intercept=relation[0],fixed_residual_slope=relation[1],full_fit_error=model['full_fit_max_abs_difference']))
fig11_table=pd.concat([fig11_physical.assign(model_version=v) for v in VERSIONS]+[pd.DataFrame(partial)],ignore_index=True)
print('Cell 4 PASS; stored full-model replay and frozen Stage-I diagnostic; day18 out-of-window')


In [ ]:
# Cell 5 — physical-size plotting/QC, no formal acceptance.
capture(5)
COLORS = {'ERA5': 'black', 'Hot17': 'firebrick', 'Middle17': '#1f77b4', 'Cold17': 'seagreen'}
LABELS = {'NCHN_tp': 'TP (mm day⁻¹)', 'z500_anom_NCHN': 'Z500 anomaly (gpm)',
          'sm_avg': 'SM (m³ m⁻³)', 'SHF_Avg': 'SHF (W m⁻²)',
          'NCVI_concurrent': 'NCVI anomaly (PVU)', 'TCC_Avg': 'TCC (fraction)',
          'partial_fitted_Tmax': 'SM–SHF diagnostic (°C)'}
STYLE = {'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans'],
         'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 9,
         'xtick.labelsize': 8, 'ytick.labelsize': 8,
         'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.bbox': None}
def limits(values):
    a = np.asarray(values, float)
    a = a[np.isfinite(a)]
    if not len(a):
        raise ValueError('No finite values for panel range')
    lo, hi = float(a.min()), float(a.max())
    pad = max((hi-lo)*0.10, abs(lo)*0.01, 0.001)
    locator = MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10])
    ticks = locator.tick_values(lo-pad, hi+pad)
    return (float(ticks[0]), float(ticks[-1])), ticks

def legend(fig, reference=True):
    names = ('ERA5', *GROUPS) if reference else GROUPS
    handles = [Line2D([0], [0], color=COLORS[g], marker='o', lw=1.4, label=g) for g in names]
    fig.legend(handles=handles, loc='lower center', ncol=len(names), frameon=False,
               bbox_to_anchor=(0.5, 0.005), fontsize=8)

def style(ax):
    ax.grid(True, linestyle=':', alpha=0.2, linewidth=0.4)
    ax.tick_params(width=0.7, length=3)
    for spine in ax.spines.values():
        spine.set_color('0.5'); spine.set_linewidth(0.6)

def timeseries_figure(df, factors, window, version=None):
    rows = len(factors)
    fig, axes = plt.subplots(rows, 2, figsize=(6.5, 9.0 if rows == 5 else 10.5), dpi=600)
    data = df if version is None else df[df.model_version == version]
    end = '2023-06-24' if window == 'Total' else '2023-06-18'
    # Pool both inits and both parameterizations for identical row scales.
    for row, factor in enumerate(factors):
        ylim, ticks = limits(df[df.factor == factor].value)
        for col, init in enumerate(INITS):
            ax = axes[row, col]
            s = data[(data.init_date == init) & (data.factor == factor)].copy()
            s['date'] = pd.to_datetime(s.date)
            for group in ('ERA5', *GROUPS):
                g = s[s.group == group].sort_values('date')
                if g.value.notna().all():
                    # Calendar-day aggregates are shown at the BJT day midpoint.
                    ax.plot(g.date + pd.Timedelta(hours=12), g.value, color=COLORS[group], marker='o', ms=2.6, lw=1.3)
                elif group == 'ERA5' and g.value.isna().all():
                    ax.text(0.02, 0.03, 'ERA5 unavailable', transform=ax.transAxes, fontsize=7, color='0.4')
                else:
                    raise ValueError('Incomplete process series cannot be silently omitted')
            ax.set_ylim(ylim); ax.set_yticks(ticks)
            ax.set_xlim(pd.Timestamp('2023-06-14'), pd.Timestamp(end) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1))
            ax.xaxis.set_major_locator(mdates.DayLocator(interval=2 if window == 'Total' else 1))
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
            if window == 'Stage-I':
                ax.axvline(pd.Timestamp('2023-06-18 00:00'), color='0.4', linestyle='--', lw=0.8)
            if window == 'Stage-I':
                ax.axvspan(pd.Timestamp('2023-06-18'), pd.Timestamp('2023-06-19'), color='0.9', alpha=0.5, zorder=-1)
            ax.set_title(f'({chr(97+row*2+col)}) {pd.Timestamp(init).day} Jun', loc='left')
            ax.set_ylabel(LABELS[factor] if col == 0 else '')
            style(ax)
    legend(fig, reference=bool((data.group=='ERA5').any()))
    fig.suptitle('PRELIMINARY / DIAGNOSTIC — '+ ('Fig.5 Total' if window=='Total' else 'Fig.11 Stage-I '+str(version)), fontsize=9)
    fig.text(0.5,0.045,'NCVI: S2S only; historical strict replay FAIL; ERA5 NCVI unverified',ha='center',fontsize=7)
    fig.subplots_adjust(left=0.15, right=0.98, top=0.91, bottom=0.09, wspace=0.28, hspace=0.60)
    return fig, axes.ravel().tolist()

def figure_qc(fig, axes):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    sizes = np.array([[a.get_position().width, a.get_position().height] for a in axes])
    if not np.allclose(sizes, sizes[0], rtol=0, atol=1e-10):
        raise ValueError('Unequal process-panel axes boxes')
    texts = list(fig.texts)
    for ax in axes:
        texts += [ax._left_title, ax.xaxis.label, ax.yaxis.label] + list(ax.texts) + ax.get_xticklabels() + ax.get_yticklabels()
    boxes = [(t.get_text(), t.get_window_extent(renderer)) for t in texts if t.get_visible() and t.get_text()]
    for label, b in boxes:
        if b.x0 < 0 or b.y0 < 0 or b.x1 > fig.bbox.width or b.y1 > fig.bbox.height:
            raise ValueError(f'Clipped text in process figure: {label}')
    for i, (label, a) in enumerate(boxes):
        for other, b in boxes[i+1:]:
            if min(a.x1,b.x1)-max(a.x0,b.x0)>1 and min(a.y1,b.y1)-max(a.y0,b.y0)>1:
                raise ValueError(f'Overlapping process labels: {label}; {other}')
    return {'equal_boxes': True, 'text_extent_qc': 'PASS', 'width_inches': fig.get_figwidth(),
            'height_inches': fig.get_figheight(), 'axes_count': len(axes)}

built=[];layout_qc=[]
with plt.rc_context(STYLE):
 fig,axes=timeseries_figure(fig5_table,FACTORS5,'Total');built.append(('Fig5_Total_DIAGNOSTIC',fig,axes))
 for version in VERSIONS:
  fig,axes=timeseries_figure(fig11_table,('partial_fitted_Tmax',*FACTORS11),'Stage-I',version)
  built.append(('Fig11_StageI_'+version+'_DIAGNOSTIC',fig,axes))
 for name,fig,axes in built:
  layout_qc.append(dict(filename=name,**figure_qc(fig,axes)))
  display(fig)
print('Cell 5 layout QC PASS; all figures DIAGNOSTIC, 6.5 inch / PNG600 dpi / vector PDF')


In [ ]:
# Cell 6 — independent export; no history/status modification.
capture(6)
if set(SNAPSHOTS)!={1,2,3,4,5,6}:raise RuntimeError('Run all cells in order')
if OUTPUT.exists():raise ValueError('Output exists; never overwrite')
if original_run_hashes!={str(p.relative_to(NCVI_RUN)):sha256(p) for p in NCVI_RUN.rglob('*') if p.is_file()}:raise ValueError('Original run changed')
for key,item in hashes.items():
 if sha256(item['path'])!=item['sha256']:raise ValueError('Input changed')
OUTPUT.mkdir(parents=True,exist_ok=False)
tables={'table_fig5_total_DIAGNOSTIC.csv':fig5_table,'table_fig11_stageI_DIAGNOSTIC.csv':fig11_table,
'v12_daily_stage_qc.csv':physical_qc,'current_groups.csv':current_groups,'partial_fitted_audit.csv':pd.DataFrame(audit),
'layout_qc.csv':pd.DataFrame(layout_qc)}
for name,table in tables.items():table.to_csv(OUTPUT/name,index=False)
for name,fig,axes in built:
 fig.savefig(OUTPUT/(name+'.png'),dpi=600)
 fig.savefig(OUTPUT/(name+'.pdf'))
snapshot=json.dumps(SNAPSHOTS,indent=2,ensure_ascii=False)
(OUTPUT/'executed_cells.json').write_text(snapshot,encoding='utf-8')
report={'status':'PRELIMINARY_DIAGNOSTIC','formal_process_allowed':False,'reference_commit':'713025b','execution_git_head':None,
'inputs':hashes,'original_ncvi_run_sha256':original_run_hashes,'original_ncvi_manifest':nm,
'daily_internal_aggregation_qc':{'status':'PASS','max_difference':float(internal_error)},
'historical_mlr_strict_qc_preserved':nm['s2s_stage_replay_qc'],'era5_ncvi':'OMITTED_UNVERIFIED',
'physical_units':{'NCHN_tp':'mm per source daily interval','sm_avg':'m3 m-3','SHF_Avg':'W m-2','z500_anom_NCHN':'gpm','TCC_Avg':'fraction','NCVI_concurrent':'PVU'},
'methods_notes':['v12 Tmax arithmetic grid mean; legacy area-weighted definition differs','TCC source interval 08:00–08:00 BJT; source date labels retained','day18 partial fit is frozen Stage-I out-of-window projection','raw/residualized diagnostic differences are parameterization sensitivity'],
'outputs':{p.name:sha256(p) for p in OUTPUT.iterdir() if p.is_file()}}
(OUTPUT/'diagnostic_process_manifest.json').write_text(json.dumps(report,indent=2,default=str))
print('Cell 6 DIAGNOSTIC exports complete:',OUTPUT,'formal_process_allowed=False; original run unchanged')
